# 06 Baseline Comparisons

Evaluate classical image-based anomaly detectors for both spectrogram datasets:

- `data/02_spectrograms_150x100px_dataset`
- `data/03_ma_dataset`

Each baseline trains only on nominal training images. Thresholds are selected on a labeled validation split and then frozen for the held-out test split. When a dataset provides a single labeled evaluation folder/file, that labeled pool is split deterministically into validation and test partitions.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

In [2]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'data').exists() else CWD.parent

BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'

IMAGE_SIZE = (150, 100)
PCA_COMPONENTS = 32
SEED = 42

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5

TURNING_MANIFEST = (
    REPO_ROOT
    / 'reports'
    / 'manifests'
    / 'turning_split_seed42.csv'
)

BROACH_MANIFEST = (
    REPO_ROOT
    / 'reports'
    / 'manifests'
    / 'broach_dataset_split_seed42.csv'
)

DATASETS = [
    {
        'name': 'turning',
        'manifest_path': TURNING_MANIFEST,
        'source_dataset': 'turning',
    },
    {
        'name': 'broach_dataset',
        'manifest_path': BROACH_MANIFEST,
        'source_dataset': 'broach_dataset',
    },
]

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def load_turning_manifest(config: dict) -> pd.DataFrame:
    manifest = pd.read_csv(config["manifest_path"])

    manifest = manifest[
        manifest["source_dataset"] == config["source_dataset"]
    ].copy()

    manifest["dataset"] = manifest["source_dataset"]
    manifest["target"] = (manifest["label"] == "chatter").astype(int)

    return manifest[
        [
            "dataset",
            "sample_id",
            "split",
            "label",
            "target",
            "image_path",
        ]
    ]

def load_broach_manifest(config: dict) -> pd.DataFrame:
    manifest = pd.read_csv(config["manifest_path"])

    manifest = manifest.copy()
    manifest["dataset"] = "broach_dataset"
    manifest["sample_id"] = (
        manifest["filename"]
        .str.replace(".png", "", regex=False)
    )

    manifest["target"] = manifest["label"]

    manifest["label"] = manifest["target"].map({
        0: "normal",
        1: "anomaly"
    })

    manifest["image_path"] = manifest.apply(
        lambda row:
            f"data/03_broach_dataset/train/{row['filename']}"
            if row["split"] == "train"
            else f"data/03_broach_dataset/test/{row['filename']}",
        axis=1,
    )

    return manifest[
        [
            "dataset",
            "sample_id",
            "split",
            "label",
            "target",
            "image_path",
        ]
    ]


def build_dataset_manifest(config: dict) -> pd.DataFrame:

    if config["name"] == "turning":
        return load_turning_manifest(config)

    if config["name"] == "broach_dataset":
        return load_broach_manifest(config)

    raise ValueError(f"Unknown dataset: {config['name']}")


In [4]:
dataset_manifests = {config['name']: build_dataset_manifest(config) for config in DATASETS}
for dataset_name, dataset_manifest in dataset_manifests.items():
    print(dataset_name)
    display(dataset_manifest.groupby(['split', 'label']).size().rename('n').reset_index())

turning


,split,label,n
0,test,chatter,27
1,test,no_chatter,48
2,train,no_chatter,472
3,validation,chatter,34
4,validation,no_chatter,70


broach_dataset


,split,label,n
0,test,anomaly,22
1,test,normal,2478
2,train,normal,5000
3,validation,anomaly,23
4,validation,normal,2477


In [5]:
def open_resized_image(image_path: Path, mode: str) -> np.ndarray:
    image = Image.open(image_path).convert(mode).resize(IMAGE_SIZE)
    return np.asarray(image, dtype='float32') / 255.0


def image_vector(image_path: Path) -> np.ndarray:
    return open_resized_image(image_path, 'L').reshape(-1)


def image_descriptor(image_path: Path) -> np.ndarray:
    arr = open_resized_image(image_path, 'L')
    grad_y, grad_x = np.gradient(arr)
    hist, _ = np.histogram(arr, bins=32, range=(0.0, 1.0), density=True)
    percentiles = np.percentile(arr, [1, 5, 25, 50, 75, 95, 99])
    summary = np.array([
        arr.mean(),
        arr.std(),
        arr.min(),
        arr.max(),
        np.abs(grad_x).mean(),
        np.abs(grad_y).mean(),
        np.sqrt(grad_x ** 2 + grad_y ** 2).mean(),
    ], dtype='float32')
    return np.concatenate([
        summary,
        percentiles.astype('float32'),
        hist.astype('float32'),
        arr.mean(axis=0),
        arr.mean(axis=1),
        arr.std(axis=0),
        arr.std(axis=1),
    ]).astype('float32')


def load_matrix(rows: pd.DataFrame, extractor) -> np.ndarray:
    values = [extractor(REPO_ROOT / image_path) for image_path in rows['image_path']]
    if not values:
        raise ValueError('No images loaded.')
    return np.stack(values, axis=0)

In [6]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        threshold = float(np.max(scores))
        return {
            'threshold': threshold,
            'validation_f1': 0.0,
            'validation_precision': 0.0,
            'validation_recall': 0.0,
        }

    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {
        'threshold': float(thresholds[best_idx]),
        'validation_f1': float(f1[best_idx]),
        'validation_precision': float(precision[best_idx]),
        'validation_recall': float(recall[best_idx]),
    }


def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def append_score_frame(
    score_frames: list[pd.DataFrame],
    rows: pd.DataFrame,
    method: str,
    score_values: np.ndarray,
    outer_fold: int | None = None,
) -> None:

    score_frames.append(pd.DataFrame({
        'dataset': rows['dataset'].to_numpy(),
        'method': method,
        'sample_id': rows['sample_id'].to_numpy(),
        'split': rows['split'].to_numpy(),
        'label': rows['label'].to_numpy(),
        'target': rows['target'].to_numpy(),
        'score_value': score_values,
        'outer_fold': outer_fold,
    }))

In [10]:
def run_descriptor_baselines(
    train_nominal: pd.DataFrame,
    evaluation_fold: pd.DataFrame,
) -> dict:

    X_train = load_matrix(train_nominal, image_descriptor)
    X_eval = load_matrix(evaluation_fold, image_descriptor)

    scaler = StandardScaler().fit(X_train)

    X_train = scaler.transform(X_train)
    X_eval = scaler.transform(X_eval)

    ocsvm = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05,
    ).fit(X_train)

    iforest = IsolationForest(
        random_state=SEED,
        contamination="auto",
    ).fit(X_train)

    return {
        "one_class_svm_image_features": (
            -ocsvm.decision_function(X_eval)
        ),
        "isolation_forest_image_features": (
            -iforest.decision_function(X_eval)
        ),
    }

In [11]:
def run_pca_reconstruction_baseline(
    train_nominal: pd.DataFrame,
    evaluation_fold: pd.DataFrame,
) -> dict:

    X_train = load_matrix(train_nominal, image_vector)
    X_eval = load_matrix(evaluation_fold, image_vector)

    n_components = min(
        PCA_COMPONENTS,
        X_train.shape[0],
        X_train.shape[1],
    )

    pca = PCA(
        n_components=n_components,
        random_state=SEED,
        svd_solver="randomized",
    ).fit(X_train)

    recon = pca.inverse_transform(
        pca.transform(X_eval)
    )

    scores = np.mean(
        (X_eval - recon) ** 2,
        axis=1,
    )

    return {
        "pca_image_reconstruction": scores
    }

In [19]:
def run_baselines_for_dataset(
    dataset_name: str,
    manifest: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:

    train_nominal = manifest[
        (manifest["split"] == "train")
        & (manifest["target"] == 0)
    ].reset_index(drop=True)

    evaluation_pool = manifest[
        manifest["split"].isin(["validation", "test"])
    ].reset_index(drop=True)

    if train_nominal.empty:
        raise ValueError(
            f"{dataset_name}: no nominal training samples found."
        )

    metric_rows = []
    score_frames = []
    thresholds = {}

    outer_cv = StratifiedKFold(
        n_splits=N_OUTER_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    X_outer = np.arange(len(evaluation_pool))
    y_outer = evaluation_pool["target"].to_numpy()

    for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_cv.split(X_outer, y_outer),
        start=1,
    ):

        outer_train = (
            evaluation_pool
            .iloc[outer_train_idx]
            .reset_index(drop=True)
        )

        outer_test = (
            evaluation_pool
            .iloc[outer_test_idx]
            .reset_index(drop=True)
        )

        inner_cv = StratifiedKFold(
            n_splits=N_INNER_FOLDS,
            shuffle=True,
            random_state=SEED,
        )

        descriptor_thresholds = {
            "one_class_svm_image_features": [],
            "isolation_forest_image_features": [],
        }

        pca_thresholds = []

        X_inner = np.arange(len(outer_train))
        y_inner = outer_train["target"].to_numpy()

        #
        # Inner CV: Threshold-Selektion
        #
        for _, inner_val_idx in inner_cv.split(
            X_inner,
            y_inner,
        ):

            inner_validation = (
                outer_train
                .iloc[inner_val_idx]
                .reset_index(drop=True)
            )

            y_inner_val = (
                inner_validation["target"]
                .to_numpy()
            )

            descriptor_scores = run_descriptor_baselines(
                train_nominal,
                inner_validation,
            )

            for method, scores in descriptor_scores.items():

                selected = select_best_f1_threshold(
                    y_inner_val,
                    scores,
                )

                descriptor_thresholds[method].append(
                    selected["threshold"]
                )

            pca_scores = run_pca_reconstruction_baseline(
                train_nominal,
                inner_validation,
            )

            selected = select_best_f1_threshold(
                y_inner_val,
                pca_scores["pca_image_reconstruction"],
            )

            pca_thresholds.append(
                selected["threshold"]
            )

        #
        # Median Thresholds aus Inner CV
        #
        final_thresholds = {
            method: float(np.median(values))
            for method, values in descriptor_thresholds.items()
        }

        final_thresholds["pca_image_reconstruction"] = float(
            np.median(pca_thresholds)
        )

        thresholds[f"outer_fold_{outer_fold}"] = final_thresholds

        #
        # Outer Test Evaluation
        #
        y_test = outer_test["target"].to_numpy()

        descriptor_scores = run_descriptor_baselines(
            train_nominal,
            outer_test,
        )

        for method, scores in descriptor_scores.items():

            metrics = evaluate_at_threshold(
                y_test,
                scores,
                final_thresholds[method],
            )

            metric_rows.append({
                "dataset": dataset_name,
                "outer_fold": outer_fold,
                "method": method,
                "score": "anomaly_score",
                "threshold": final_thresholds[method],
                **metrics,
            })

            append_score_frame(
                score_frames,
                outer_test,
                method,
                scores,
                outer_fold,
            )

        pca_scores = run_pca_reconstruction_baseline(
            train_nominal,
            outer_test,
        )

        metrics = evaluate_at_threshold(
            y_test,
            pca_scores["pca_image_reconstruction"],
            final_thresholds["pca_image_reconstruction"],
        )

        metric_rows.append({
            "dataset": dataset_name,
            "outer_fold": outer_fold,
            "method": "pca_image_reconstruction",
            "score": "reconstruction_mse",
            "threshold": final_thresholds[
                "pca_image_reconstruction"
            ],
            **metrics,
        })

        append_score_frame(
            score_frames,
            outer_test,
            "pca_image_reconstruction",
            pca_scores["pca_image_reconstruction"],
            outer_fold,
        )

    metrics = pd.DataFrame(metric_rows)
    scores = pd.concat(score_frames, ignore_index=True)

    return metrics, scores, thresholds

In [20]:
all_metric_frames = []
all_score_frames = []
all_thresholds = {}

for dataset_name, dataset_manifest in dataset_manifests.items():

    metrics, scores, thresholds = run_baselines_for_dataset(
        dataset_name,
        dataset_manifest,
    )

    metrics_path = TABLE_DIR / f"metrics_{dataset_name}_nested_cv.csv"
    scores_path = BASELINE_DIR / f"baseline_scores_{dataset_name}_nested_cv.csv"
    threshold_path = THRESHOLD_DIR / f"baseline_thresholds_{dataset_name}_nested_cv.json"

    metrics.to_csv(metrics_path, index=False)
    scores.to_csv(scores_path, index=False)

    with threshold_path.open("w") as f:
        json.dump(thresholds, f, indent=2)

    print(f"Wrote {metrics_path}")
    print(f"Wrote {scores_path}")
    print(f"Wrote {threshold_path}")

    all_metric_frames.append(metrics)
    all_score_frames.append(scores)
    all_thresholds[dataset_name] = thresholds

baseline_metrics = pd.concat(
    all_metric_frames,
    ignore_index=True,
)

baseline_scores = pd.concat(
    all_score_frames,
    ignore_index=True,
)

combined_metrics_path = (
    TABLE_DIR
    / "metrics_nested_cv_all_datasets.csv"
)

combined_scores_path = (
    BASELINE_DIR
    / "baseline_scores_nested_cv_all_datasets.csv"
)

combined_threshold_path = (
    THRESHOLD_DIR
    / "baseline_thresholds_nested_cv_all_datasets.json"
)

baseline_metrics.to_csv(
    combined_metrics_path,
    index=False,
)

baseline_scores.to_csv(
    combined_scores_path,
    index=False,
)

with combined_threshold_path.open("w") as f:
    json.dump(all_thresholds, f, indent=2)

print(f"Wrote {combined_metrics_path}")
print(f"Wrote {combined_scores_path}")
print(f"Wrote {combined_threshold_path}")

baseline_metrics

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_nested_cv.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\baselines\baseline_scores_turning_nested_cv.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\thresholds\baseline_thresholds_turning_nested_cv.json
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broach_dataset_nested_cv.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\baselines\baseline_scores_broach_dataset_nested_cv.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\thresholds\baseline_thresholds_broach_dataset_nested_cv.json
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_nested_cv_all_datasets.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\baselines\baseline_scores_nested_cv_all_datasets.csv
Wrote C:\Users\Zeleny\

,dataset,outer_fold,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,turning,1,one_class_svm_image_features,anomaly_score,0.250410,1.000000,0.960000,1.000000,0.923077,23,0,1,12
1,turning,1,isolation_forest_image_features,anomaly_score,0.078702,0.988588,0.916667,1.000000,0.846154,23,0,2,11
2,turning,1,pca_image_reconstruction,reconstruction_mse,0.001478,0.985577,0.869565,1.000000,0.769231,23,0,3,10
3,turning,2,one_class_svm_image_features,anomaly_score,0.250410,0.959402,0.909091,1.000000,0.833333,24,0,2,10
4,turning,2,isolation_forest_image_features,anomaly_score,0.057502,0.993590,0.960000,0.923077,1.000000,23,1,0,12
5,turning,2,pca_image_reconstruction,reconstruction_mse,0.001496,0.955982,0.869565,0.909091,0.833333,23,1,2,10
6,turning,3,one_class_svm_image_features,anomaly_score,0.256667,0.957341,0.880000,0.846154,0.916667,22,2,1,11
7,turning,3,isolation_forest_image_features,anomaly_score,0.079251,0.965242,0.869565,0.909091,0.833333,23,1,2,10
8,turning,3,pca_image_reconstruction,reconstruction_mse,0.001478,0.936436,0.888889,0.800000,1.000000,21,3,0,12
9,turning,4,one_class_svm_image_features,anomaly_score,0.355317,1.000000,0.909091,1.000000,0.833333,24,0,2,10
